In [1]:
import pandas as pd
import numpy as np
import requests

# Download OPSD 60-min data
url = "https://data.open-power-system-data.org/time_series/2020-10-06/time_series_60min_singleindex.csv"
opsd = pd.read_csv(url, parse_dates=["utc_timestamp"])

# Filter German load (same column as in your project)
load_col = "DE_load_actual_entsoe_transparency"
df = opsd[["utc_timestamp", load_col]].rename(columns={load_col: "load_mw"}).dropna()
df = df.set_index("utc_timestamp").loc["2015-01-01":]

# Resample to daily mean in GW
daily = df.resample("D").mean()
daily["load_gw"] = daily["load_mw"] / 1000.0
daily = daily[["load_gw"]]

print(daily.shape)
daily.head()

(2100, 1)


,load_gw
utc_timestamp,
2015-01-01 00:00:00+00:00,45.346542
2015-01-02 00:00:00+00:00,51.941167
2015-01-03 00:00:00+00:00,46.564750
2015-01-04 00:00:00+00:00,45.082500
2015-01-05 00:00:00+00:00,55.246667


In [2]:
from sklearn.preprocessing import MinMaxScaler

def make_sliding_windows(series: pd.Series, window_size: int, horizon: int = 1):
    values = series.values.reshape(-1, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    values_scaled = scaler.fit_transform(values)

    X_list, y_list = [], []
    for i in range(len(values_scaled) - window_size - horizon + 1):
        X_list.append(values_scaled[i : i + window_size])
        y_list.append(values_scaled[i + window_size + horizon - 1, 0])

    X = np.array(X_list)  # (samples, timesteps, features=1)
    y = np.array(y_list)
    return X, y, scaler

def split_sequences_time_based(X, y, test_fraction: float = 0.2):
    n_samples = X.shape[0]
    n_test = int(n_samples * test_fraction)
    n_train = n_samples - n_test
    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]
    return X_train, X_test, y_train, y_test

window_size = 30  # past 30 days
X, y, scaler = make_sliding_windows(daily["load_gw"], window_size=window_size, horizon=1)
X_train, X_test, y_train, y_test = split_sequences_time_based(X, y, test_fraction=0.2)

print("X shape:", X.shape)
print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

X shape: (2070, 30, 1)
Train samples: 1656
Test samples: 414


In [3]:
from sklearn.preprocessing import MinMaxScaler

def make_sliding_windows(series: pd.Series, window_size: int, horizon: int = 1):
    values = series.values.reshape(-1, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    values_scaled = scaler.fit_transform(values)

    X_list, y_list = [], []
    for i in range(len(values_scaled) - window_size - horizon + 1):
        X_list.append(values_scaled[i : i + window_size])
        y_list.append(values_scaled[i + window_size + horizon - 1, 0])

    X = np.array(X_list)  # (samples, timesteps, features=1)
    y = np.array(y_list)
    return X, y, scaler

def split_sequences_time_based(X, y, test_fraction: float = 0.2):
    n_samples = X.shape[0]
    n_test = int(n_samples * test_fraction)
    n_train = n_samples - n_test
    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]
    return X_train, X_test, y_train, y_test

window_size = 30  # past 30 days
X, y, scaler = make_sliding_windows(daily["load_gw"], window_size=window_size, horizon=1)
X_train, X_test, y_train, y_test = split_sequences_time_based(X, y, test_fraction=0.2)

print("X shape:", X.shape)
print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

X shape: (2070, 30, 1)
Train samples: 1656
Test samples: 414


In [4]:
import tensorflow as tf
from tensorflow import keras

def build_univariate_lstm(input_shape):
    model = keras.Sequential(
        [
            keras.layers.Input(shape=input_shape),
            keras.layers.LSTM(64, return_sequences=False),
            keras.layers.Dense(32, activation="relu"),
            keras.layers.Dense(1),
        ]
    )

    model.compile(
        loss="mse",
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        metrics=["mae"],
    )
    return model

model = build_univariate_lstm(input_shape=(X_train.shape[1], X_train.shape[2]))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,009 (74.25 KB)

 Trainable params: 19,009 (74.25 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True
    )
]

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    verbose=1,
    shuffle=False,  # keep time order
    callbacks=callbacks,
)

Epoch 1/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 0.0692 - mae: 0.2170 - val_loss: 0.0440 - val_mae: 0.1823
Epoch 2/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0431 - mae: 0.1825 - val_loss: 0.0447 - val_mae: 0.1763
Epoch 3/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0430 - mae: 0.1813 - val_loss: 0.0444 - val_mae: 0.1751
Epoch 4/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0426 - mae: 0.1805 - val_loss: 0.0438 - val_mae: 0.1746
Epoch 5/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0421 - mae: 0.1795 - val_loss: 0.0431 - val_mae: 0.1734
Epoch 6/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0417 - mae: 0.1781 - val_loss: 0.0422 - val_mae: 0.1715
Epoch 7/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0404 - mae: 0.1754 - val_loss: 0.0407 - val_mae: 0.1662
Epoch 8/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0380 - mae: 0.1689 - val_loss: 0.0371 - val_mae: 0.1521
Epoch 9/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - lo

In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Predict on test set (scaled)
y_pred_scaled = model.predict(X_test).flatten()

# Inverse scaling back to GW
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_inv = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

mae = mean_absolute_error(y_test_inv, y_pred_inv)
rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
bias = float(np.mean(y_pred_inv - y_test_inv))

print("LSTM daily - MAE:", mae)
print("LSTM daily - RMSE:", rmse)
print("LSTM daily - Bias:", bias)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
LSTM daily - MAE: 1.299733049721342
LSTM daily - RMSE: 2.07791321033945
LSTM daily - Bias: 0.47093309608250616


In [7]:
import pandas as pd

metrics_df = pd.DataFrame(
    [
        {
            "model": "lstm_daily",
            "MAE": mae,
            "RMSE": rmse,
            "Bias": bias,
        }
    ]
)

from google.colab import files

metrics_df.to_csv("lstm_daily_metrics.csv", index=False)
files.download("lstm_daily_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# Match test timestamps: last len(y_test_inv) days
test_index = daily.index[-len(y_test_inv):]

forecasts_df = pd.DataFrame(
    {
        "utc_timestamp": test_index,
        "actual": y_test_inv,
        "lstm_daily": y_pred_inv,
    }
).set_index("utc_timestamp")

forecasts_df.to_csv("lstm_daily.csv")
files.download("lstm_daily.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>